# Battle Scanner — D1 GPU Embeddings (Colab)

Embed the downsized `wh40k_embed_bundle` on a GPU, then download `embeddings.npz`.
Mirror of `scripts/curation/embed_gpu.py`. **Runtime → Change runtime type → GPU.**

Steps: drop `wh40k_embed_bundle.tar` in your Google Drive, run all cells, download the npz,
then locally: `fiftyone_env/bin/python scripts/curation/load_embeddings.py --npz embeddings.npz`.


In [ ]:
# 1. deps (Colab already has torch+CUDA)
!pip -q install transformers


In [ ]:
# 2. mount Drive and point at the uploaded bundle tar
from google.colab import drive
drive.mount('/content/drive')
TAR = '/content/drive/MyDrive/wh40k_embed_bundle.tar'  # <- edit if you put it elsewhere
MODEL = 'facebook/dinov2-large'  # open, no gating. Swap to a DINOv3 ckpt for the retrieval gallery later.


In [ ]:
# 3. extract
import tarfile, pathlib
BUNDLE = pathlib.Path('/content/bundle')
BUNDLE.mkdir(exist_ok=True)
with tarfile.open(TAR) as t: t.extractall(BUNDLE)
print(sorted(p.name for p in BUNDLE.iterdir()))


In [ ]:
# 4. embed (CLS token, L2-normalised, fp16)
import csv, numpy as np, torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel

img_dir = BUNDLE/'images'
rows = [(int(r['idx']), r['filepath']) for r in csv.DictReader(open(BUNDLE/'manifest.csv')) if r['status']=='ok']
print(len(rows), 'images')
proc = AutoImageProcessor.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).cuda().eval()

class DS(Dataset):
    def __len__(self): return len(rows)
    def __getitem__(self, i):
        idx, fp = rows[i]
        im = Image.open(img_dir/f'{idx:06d}.jpg').convert('RGB')
        return idx, fp, proc(images=im, return_tensors='pt')['pixel_values'][0]
def collate(b): return [x[0] for x in b], [x[1] for x in b], torch.stack([x[2] for x in b])
dl = DataLoader(DS(), batch_size=64, num_workers=2, collate_fn=collate)

idxs, fps, embs = [], [], []
with torch.no_grad():
    for k,(bi,bf,pix) in enumerate(dl):
        cls = model(pixel_values=pix.cuda()).last_hidden_state[:,0]
        cls = torch.nn.functional.normalize(cls, dim=1)
        embs.append(cls.cpu().to(torch.float16).numpy()); idxs += bi; fps += bf
        if k % 40 == 0: print(k*64, '/', len(rows))
E = np.concatenate(embs); print('embeddings', E.shape)


In [ ]:
# 5. save + download
import numpy as np
np.savez_compressed('embeddings.npz', idx=np.array(idxs,dtype=np.int32),
                    embeddings=E, filepaths=np.array(fps), model=MODEL)
from google.colab import files; files.download('embeddings.npz')
